<a href="https://colab.research.google.com/github/lmknijn/jeweled_style/blob/main/golden_silver_lines_latincy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install language model, prerequisites

In [ ]:
# install language models (Latin/Greek)
!pip install https://huggingface.co/latincy/la_core_web_trf/resolve/main/la_core_web_trf-3.9.5-py3-none-any.whl
!pip install https://huggingface.co/latincy/gr_dep_web_trf/resolve/main/gr_dep_web_trf-3.8.4-py3-none-any.whl

In [ ]:
import spacy
nlp = spacy.load("la_core_web_trf") # or "gr_dep_web_trf"

In [ ]:
# utils
import os
import json
import requests
import unicodedata as ud

# for analysis
from collections import Counter
import pandas as pd
import numpy as np
from tqdm import tqdm

# for output
from IPython.display import HTML
import seaborn as sns
from matplotlib import pyplot as plt

Convert text file to right format

In [ ]:
# read uploaded CSV file, change 'text' to abbreviated work name
text = pd.read_csv('file_name.csv')

In [ ]:
# split lines into two columns, adjust according to text format
import re

# create pattern to split everything between <> and the rest into two columns
pattern = r'^<(.*?)>\s*(.*)$'

extracted_data = sil['info'].str.extract(pattern)
text = pd.DataFrame({
    'info': extracted_data[0],
    'text': extracted_data[1]
})

display(text)

In [ ]:
# remove rows where 'info' or 'text' is NaN (but check in text ed why the extraction failed)
text.dropna(subset=['info', 'text'], inplace=True)
text.reset_index(drop=True, inplace=True)
display(text)

In [ ]:
# split 'info' column into book and line number
text = pd.DataFrame({
    'info': extracted_data[0],
    'text': extracted_data[1]
})

# fill in fixed names for 'author' and 'work'
text['author'] = 'valerius flaccus'
text['work'] = 'argonautica'

# adjust formula if necessary
extracted_book_line = text['info'].str.extract(r'(\d+)\.(\d+)')

# create columns for 'book' and 'line'
text['book'] = extracted_book_line[0]
text['line'] = extracted_book_line[1]

# convert 'book' and 'line' values to nullable integer type, errors become NaN
text['book'] = pd.to_numeric(sil['book'], errors='coerce').astype(pd.Int64Dtype())
text['line'] = pd.to_numeric(sil['line'], errors='coerce').astype(pd.Int64Dtype())

# create 'line_id' column
text['line_id'] = text.apply(lambda row: f"{row['book']:02d}_{row['line']:04d}" if pd.notna(row['book']) and pd.notna(row['line']) else None, axis=1)

# reorder columns
text = text[['author', 'work', 'book', 'line', 'line_id', 'text']]

display(text)

Run Latincy on text

In [ ]:
# create list of dictionaries
token_rows = []

# make sure 'text' column contains only strings, replacing NaN with empty string
processed_text = text['text'].fillna('').astype(str)

# recreate the 'docs_generator', processing the 'text' column
docs_generator = nlp.pipe(processed_text)

# iterate over the spacy doc objects and get their corresponding book, line, line_id
for i, spacy_doc in tqdm(enumerate(docs_generator), total=len(coll)):
  # get the book, line, line_id
  current_book = text.loc[i, 'book']
  current_line = text.loc[i, 'line']
  current_line_id = text.loc[i, 'line_id']

  for token in spacy_doc:
      token_rows.append({
        'book': current_book,
        'line': current_line,
        'line_id': current_line_id,
        'token': token.text,
        'lemma': token.lemma_,
        'pos': token.pos_,
        'case': token.morph.get('Case'),
        'number': token.morph.get('Number'),
        'gender': token.morph.get('Gender'),
        'person': token.morph.get('Person'),
        'dep': token.dep_
    })

# create a DataFrame from the token data
tokens = pd.DataFrame(
    token_rows,
    columns=['book', 'line', 'line_id', 'token', 'lemma', 'pos', 'case', 'number', 'gender', 'person', 'dep']
)


# Display the DataFrame
display(tokens)

In [ ]:
# plug in already tokenized .csv files here
text = pd.read_csv('text_file.csv')
display(text)

Shaping the DataFrame

In [ ]:
# remove punctuation
nopunct_tokens = text[text['pos'] != 'PUNCT']
display(nopunct_tokens)

In [ ]:
# add n of tokens per line
token_count = nopunct_tokens.groupby('line_id').agg(
    ntokens=('token', 'count')
).reset_index()

# Merge token count per line with token table
merged_table = pd.merge(token_count, no_punct_tokens, on='line_id', how='left')

display(merged_table)

# download token table as CSV file for future reference
merged_table.to_csv('text_tokenized_date.csv')

In [ ]:
# create a row for each line and lists in each column
pos_per_line = merged_table.groupby('line_id').agg(
    book=('book', 'first'),
    line=('line', 'first'),
    ntokens=('ntokens', 'first'),
    tokens=('token', list),
    lemmas=('lemma', list),
    pos=('pos', list),
    case=('case', list),
    number=('number', list),
    gender=('gender', list),
    person=('person', list),
    dep=('dep', list)
).reset_index()

display(pos_per_line)

Adding golden/silver line criteria

In [ ]:
# filter for abVAB - ADJ ADJ VERB NOUN/PROPN NOUN/PROPN
# no other ADJ, VERB, NOUN, PROP outside of sequence allowed

def abVAB_propn_filter(pos_list):
    pos_sequence = ['ADJ', 'ADJ', 'VERB', ('NOUN', 'PROPN'), ('NOUN', 'PROPN')]
    current_sequence_idx = 0
    matched_indices = [] # Store indices of matched sequence elements

    for i, pos_tag in enumerate(pos_list):
        if current_sequence_idx < len(pos_sequence):
            expected_pos = pos_sequence[current_sequence_idx]
            match = False
            if isinstance(expected_pos, tuple):
                if pos_tag in expected_pos:
                    match = True
            elif pos_tag == expected_pos:
                match = True

            if match:
                matched_indices.append(i) # Add index to matched_indices
                current_sequence_idx += 1

    # If the target sequence was not fully found, it's not a match
    if current_sequence_idx != len(pos_sequence):
        return False

    # Now, check for any other ADJ, VERB, NOUN, PROPN outside the matched sequence
    allowed_pos_types = ['ADJ', 'VERB', 'NOUN', 'PROPN']
    for i, pos_tag in enumerate(pos_list):
        if pos_tag in allowed_pos_types and i not in matched_indices:
          return False # Found an disallowed POS tag outside the sequence

    return True

pos_filtered_lines = pos_per_line[pos_per_line['pos'].apply(abVAB_propn_filter)]

display(pos_filtered_lines)

# manually analyze results
pos_filtered_lines.to_csv('text_AAVNN_date.csv')